<div align="center">

# Symbolic recovery of CRDFT reactivity descriptors: a controlled benchmark of Kolmogorov--Arnold networks, sparse identification and genetic programming

*Supporting notebook - KAN*

</div>

---

This notebook produces every KAN result in the manuscript: the fits, the symbolic exports, the
closed forms read from the network alone, and the optimizer, spline, seed, noise and
training-size sweeps. Put `reactivity_dataset.csv`, `rsc_style.py`, `figures_rsc.py` and
`requirements.txt` in the working directory, then Run All (~45-60 min on a CPU runtime).
`DRY=1` shrinks every budget for a smoke test, and `README.md` has the full instructions.

## 1. What this notebook does

CRDFT global descriptors are exact algebraic functions of the ionization potential and electron
affinity (Table 1 of the manuscript; Cell 3.3 verifies them to ~1e-15), so a recovered
expression can be compared against a known answer. The study runs in three steps: direct fits
(4.1-4.3), the logarithmic route with the closed forms read from the network alone (4.4-4.4c),
and the robustness sweeps (4.6-4.11b).

### Reading the results: three coefficients of determination

Every multiplicative descriptor is reported with three $R^2$ values, each printed with a label.
The analytic log-OLS $R^2$ measures the data (is the true relationship log-linear?), the
closed-form $R^2$ on all $N$ certifies recovery, and the KAN spline $R^2$ (standardized,
held-out subsample) measures the network's own fit.

## 2. The two structures the code has to handle

A one-hidden-layer KAN realizes $f(x_1,x_2)=\Phi(\phi_1(x_1)+\phi_2(x_2))$, which fits the
affine and univariate descriptors but cannot represent a bare product or quotient. For a
monomial $y=c\prod_i a_i^{p_i}$, $\ln y=\ln c+\sum_i p_i\ln a_i$ is linear in $\ln a_i$, so with
the library `["x"]` the edge slopes are the exponents and the intercept is $\ln c$. Each closed
form is built twice without a shared intermediate: the network route (4.4c) reads the edge
slopes, the anchor route solves log-space least squares exactly.

## 3. Computational methods

Fits use cubic splines on a grid of size 3, an Adam warm-up followed by L-BFGS, and a training
subsample; the grid is placed on the standardized training inputs before training. pykan's
default grid spans $[-1,1]$, which leaves much of the standardized input outside the spline
support, and `KAN_DEFAULT_GRID=1` reproduces that failure mode. Budgets, split sizes and the log
guard are set in Cell 3.1 and reported by Cells 3.2 and 3.3b.

> **Large-data run.** This notebook was executed on the full descriptor set. Fitting uses an `N_FIT` subsample while validation always uses **all** molecules, and the log target is guarded against near-zero atoms (which would send $\ln\omega^{+}\to-\infty$). Every closed form is built twice, from the network alone (4.4c) and from the exact log-space least-squares anchor (4.4, 4.7).

> **Reproducibility note.** Headline models are trained for `STEPS_HEAD` optimizer steps on a random `N_FIT`-molecule training subsample (fitting cost only); every reported $R^2$ and parity plot is evaluated on the **full** data set. The exact values of `STEPS_HEAD`, `N_FIT` and the dataset size are printed by the configuration and data cells below.

In [ ]:
# 3.0 rsc_style.py and figures_rsc.py ship with this repository, not PyPI; on Colab this
# cell prompts for them if they are missing.
import os as _os, importlib.util as _ilu
# Skip the install when the environment already provides everything: on a cluster it is
# built beforehand, and re-running pip can change a pinned version mid-study.
_REQUIRED = ("kan", "torch", "sympy", "numpy", "pandas", "sklearn", "matplotlib")
_missing = [m for m in _REQUIRED if _ilu.find_spec(m) is None]
if _os.environ.get("SKIP_PIP", "0") == "1" or not _missing:
    print("dependencies already satisfied; skipping pip install"
          + (" (SKIP_PIP=1)" if _os.environ.get("SKIP_PIP") == "1" else ""))
elif _os.path.exists("requirements.txt"):
    !pip -q install -r requirements.txt      # pinned versions (pykan's API changed across 0.1/0.2)
else:
    !pip -q install pykan==0.2.8 sympy scikit-learn pandas numpy torch matplotlib

import os, sys
sys.path.insert(0, os.getcwd())          # make the working directory importable

_needed = [f for f in ("rsc_style.py", "figures_rsc.py") if not os.path.exists(f)]
if _needed:
    try:                                  # Colab
        from google.colab import files
        print("Missing:", ", ".join(_needed))
        print("Select them in the file picker (both at once is fine).")
        up = files.upload()
        print("Uploaded:", ", ".join(up))
    except ImportError:                   # local Jupyter
        raise FileNotFoundError(
            f"Put {', '.join(_needed)} in the same folder as this notebook "
            f"(currently {os.getcwd()}), then re-run this cell.")

import importlib, rsc_style, figures_rsc
importlib.reload(rsc_style); importlib.reload(figures_rsc)
# Version guard. Compare as an INTEGER, not a string: with an older file the notebook
# fails half an hour into the run, or silently draws the old figure.
FIGURES_RSC_MIN = 12
_have = int(getattr(figures_rsc, "__version__", "0"))
if _have < FIGURES_RSC_MIN:
    raise ImportError(
        f"figures_rsc.py is v{_have}; this notebook needs v{FIGURES_RSC_MIN} or newer. "
        "Delete the old file from the file panel (or restart the runtime), upload the "
        "current one, and re-run this cell.")
for _fn in ("fig_trainsize", "fig03_parity", "fig07_noise"):
    if not hasattr(figures_rsc, _fn):
        raise ImportError(f"figures_rsc.py v{_have} has no {_fn}(); upload the current file.")
print(f"companion modules loaded OK (figures_rsc v{figures_rsc.__version__})")


**Cell 3.1. Imports and global configuration.** Fixes every experimental knob in one place:
the optimizer steps for the headline fits, the Adam budgets of the optimizer study, the seed
list, the noise grid and the spline hyper-parameter grid. Setting `DRY=1` shrinks all of these
so the notebook executes end-to-end in seconds for debugging.

In [ ]:
# Determinism: pin the thread count BEFORE torch is imported. Float addition is not
# associative, so the count moves the unrounded exponents; keep one value for the study.
import os
KAN_THREADS = int(os.environ.get("KAN_THREADS", "1"))   
if KAN_THREADS:
    for _v in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS",
               "NUMEXPR_NUM_THREADS"):
        os.environ[_v] = str(KAN_THREADS)

import warnings, time, platform
import numpy as np, pandas as pd, torch, sympy as sp
if KAN_THREADS:
    torch.set_num_threads(KAN_THREADS)
    try:                      # may only be set once, and only before any parallel work
        torch.set_num_interop_threads(KAN_THREADS)
    except RuntimeError:
        pass
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from kan import KAN

# Filter narrowly, not globally: a genuine pykan/torch version mismatch must stay visible.
warnings.filterwarnings("ignore", category=DeprecationWarning, module="kan")
warnings.filterwarnings("ignore", category=UserWarning, module="torch")

torch.set_default_dtype(torch.float32)
GLOBAL_SEED = 0
np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)          # the KAN seed= covers init only, not every op
os.makedirs("model", exist_ok=True)
plt.rcParams.update({"figure.dpi":110,"font.size":11,"axes.grid":True,"grid.alpha":.3,
                     "axes.spines.top":False,"axes.spines.right":False})

# DRY=1 shrinks every budget so the notebook runs end-to-end in a couple of minutes.
DRY = os.environ.get("DRY", "0") == "1"

if DRY:
    STEPS_HEAD, ADAM_WARM, N_FIT = 20, 20, 400
    SEEDS = [0, 7]
    LBFGS_OPT_STEPS, ADAM_BUDGETS = 20, [20, 40]
    GRIDK  = [(3,3), (5,3)]
    SIGMAS = [0.0, 0.2]
else:
    STEPS_HEAD = 500      # L-BFGS refinement steps for the headline fits (>=500)
    ADAM_WARM  = 300      # Adam warm-up steps before L-BFGS
    N_FIT      = 3000     # training subsample size (fit cost only; validation uses ALL)
    SEEDS      = [0,7,21,42,123]
    LBFGS_OPT_STEPS = 500          # optimizer study: L-BFGS budget
    ADAM_BUDGETS    = [500, 1000]  # optimizer study: Adam budgets
    GRIDK  = [(3,3), (5,3), (3,5), (7,3), (3,7)]   # spline sweep  (Cell 4.10)
    SIGMAS = [0.0, 0.1, 0.2, 0.4]                  # noise sweep   (Cell 4.12)

NOISE_SEEDS = SEEDS
if os.environ.get("KAN_DEFAULT_GRID","0")=="1":
    print("KAN_DEFAULT_GRID=1: spline grids left at pykan's default [-1,1] (reproduces the pitfall)")
FAILURES = []          # symbolic-recovery failures are recorded here, never swallowed

print(("DRY" if DRY else "LARGE-DATA") + " configuration | "
      f"STEPS_HEAD={STEPS_HEAD} | ADAM_WARM={ADAM_WARM} | N_FIT={N_FIT} | seeds={SEEDS}")

import importlib.metadata as _md
def _ver(p):
    try: return _md.version(p)
    except Exception: return "not installed"
print(f"\nEnvironment\n  python   {platform.python_version()}  ({platform.platform()})")
for _p in ["torch","pykan","sympy","numpy","pandas","scikit-learn","matplotlib"]:
    print(f"  {_p:<14} {_ver(_p)}")
print(f"  cuda available {torch.cuda.is_available()}")
print(f"  torch threads   {torch.get_num_threads()} (KAN_THREADS={KAN_THREADS}; pinned for reproducibility)")

# Snapping helpers, defined here so that any single cell below can run on its own. PHYS
# holds exactly the prefactors the CDFT laws use, a strong prior (raw values in 4.4b).
PHYS=[sp.Integer(1),sp.Rational(1,2),sp.Rational(1,4),sp.Rational(1,8),
      sp.Rational(1,16),sp.Integer(2),sp.Integer(3),sp.Integer(16)]
def _snap_const(v):
    c=min(PHYS+[-x for x in PHYS],key=lambda z:abs(float(z)-v))
    return c if abs(float(c)-v)<0.03*max(abs(v),1e-9) else sp.nsimplify(v,tolerance=0.01)
def _snap_half(v): return sp.Rational(round(2*float(v)),2)
snap_half=_snap_half

**Cell 3.2. Load and curate the dataset.** Reads `reactivity_dataset.csv`, keeps only
physically meaningful samples ($\eta>0$), drops non-finite rows, and prints a statistics table.
The SHA-256 of the CSV is printed, and the cell **stops** if the curated molecule count differs
from `EXPECTED_N` (131,375, Section 3.1 of the paper).

In [ ]:
# Without the dataset a synthetic stand-in runs end-to-end but does NOT reproduce the paper.
CSV_PATH = "reactivity_dataset.csv"
EXPECTED_N = int(os.environ.get("EXPECTED_N", "131375"))   # curated set of the paper (Section 3.1)
USED_SYNTHETIC = not os.path.exists(CSV_PATH)
import hashlib
DATA_SHA256 = None if USED_SYNTHETIC else hashlib.sha256(open(CSV_PATH, "rb").read()).hexdigest()

if not USED_SYNTHETIC:
    df = pd.read_csv(CSV_PATH)
else:
    warnings.warn("reactivity_dataset.csv not found -- using SYNTHETIC data. "
                  "Results below do NOT reproduce the manuscript.", RuntimeWarning)
    print("="*74); print("  WARNING: SYNTHETIC DATA IN USE -- results are NOT reproducible")
    print("="*74)
    rng=np.random.default_rng(7); N=131385
    IPg=np.clip(rng.normal(8.56,1.14,N),3.69,15.89); EAg=np.clip(rng.normal(-0.46,1.28,N),-4.45,4.57)
    etag=IPg-EAg; m=etag>0.05; IPg,EAg,etag=IPg[m],EAg[m],etag[m]; mug=-(IPg+EAg)/2; chig=(IPg+EAg)/2
    df=pd.DataFrame({"IP":IPg,"EA":EAg,"mu":mug,"chi":chig,"eta":etag,"S":1/etag,"Nmax":chig/etag,
                     "omega":mug**2/(2*etag),"omega_plus":(IPg+3*EAg)**2/(16*etag),
                     "omega_minus":(3*IPg+EAg)**2/(16*etag)})

df = df[df["eta"]>0].replace([np.inf,-np.inf],np.nan).dropna().reset_index(drop=True)
IP,EA = df["IP"].values, df["EA"].values
mu,eta = df["mu"].values, df["eta"].values

N_MOL = len(df)        # every later count is derived from this, never hard-coded
print(f"Molecules: {N_MOL}   (source: {'SYNTHETIC' if USED_SYNTHETIC else CSV_PATH})")
print(f"Fit subsample N_FIT = {N_FIT}; all {N_MOL} molecules are used for validation.")
if DATA_SHA256: print(f"CSV sha256: {DATA_SHA256}")
if not USED_SYNTHETIC and N_MOL != EXPECTED_N:
    raise RuntimeError(f"{CSV_PATH} gives {N_MOL} molecules after curation, but the paper's data set "
                       f"has {EXPECTED_N}. This is a different file; fix it (or set EXPECTED_N if intended).")
df[["IP","EA","mu","chi","eta","S","Nmax","omega","omega_plus","omega_minus"]].describe().T[["mean","std","min","max"]].round(4)

**Cell 3.3. Sanity check: the reference identities.** Confirms that each descriptor column
equals its textbook formula to floating-point precision ($\sim10^{-15}$), which is what makes
the dataset a clean *recovery* benchmark.

In [ ]:
checks = {
    "mu   = -(IP+EA)/2"        : -(IP+EA)/2            - df["mu"].values,
    "chi  = (IP+EA)/2"         : (IP+EA)/2             - df["chi"].values,
    "eta  = IP-EA"             : (IP-EA)               - df["eta"].values,
    "S    = 1/eta"             : 1/eta                 - df["S"].values,
    "Nmax = chi/eta"           : (-mu)/eta             - df["Nmax"].values,
    "omega= chi^2/2eta"        : mu**2/(2*eta)         - df["omega"].values,
    "w+   = (IP+3EA)^2/16eta"  : (IP+3*EA)**2/(16*eta) - df["omega_plus"].values,
    "w-   = (3IP+EA)^2/16eta"  : (3*IP+EA)**2/(16*eta) - df["omega_minus"].values,
}
for k,v in checks.items(): print(f"{k:26s} max|res| = {np.abs(v).max():.2e}")

**Cell 3.3b. Split and log-guard bookkeeping.** Reports how many molecules enter each fit and
how many the logarithmic guard removes. Pure NumPy: it runs in a second and fits nothing.

In [ ]:
_p = np.random.RandomState(42).permutation(N_MOL)
_tr, _te = _p[:N_FIT], _p[N_FIT:N_FIT+2000]
print(f"Split: train {len(_tr)}  test {len(_te)}  unused {N_MOL-len(_tr)-len(_te)}  "
      f"(disjoint: {len(set(_tr)&set(_te))==0}); all {N_MOL} used for closed-form R2\n")

_ATOM = {"chi": -mu, "eta": eta,
         "Aplus": np.abs(IP+3*EA), "Aminus": np.abs(3*IP+EA)}
print(f"{'descriptor':12s} {'feats':18s} {'kept':>8s} {'dropped':>8s}   min|atom|")
for _name, _feats, _ycol in [("S",["eta"],"S"), ("Nmax",["chi","eta"],"Nmax"),
                             ("omega",["chi","eta"],"omega"),
                             ("omega_plus",["Aplus","eta"],"omega_plus"),
                             ("omega_minus",["Aminus","eta"],"omega_minus")]:
    _y = df[_ycol].values; _g = (_y > 1e-12)
    for _f in _feats: _g &= np.abs(_ATOM[_f]) > 1e-9
    _mn = min(float(np.abs(_ATOM[_f]).min()) for _f in _feats)
    print(f"{_name:12s} {','.join(_feats):18s} {int(_g.sum()):8d} {int((~_g).sum()):8d}   {_mn:.3e}")

**Cell 3.4. Engine A, the general KAN and symbolic pipeline.** `kan_symbolic` builds a
train/test split, instantiates a KAN of the requested `width`, trains it, prunes inactive edges,
and calls `auto_symbolic` against a primitive `lib`. It returns the exported SymPy expression,
the spline and symbolic test $R^2$, the loss history, the wall-clock time and the model.

In [ ]:
def _subsample_idx(n, seed=42):
    p = np.random.RandomState(seed).permutation(n); return p[:N_FIT], p[N_FIT:N_FIT+2000]

def kan_symbolic(Xcols, ycol, width, lib, steps=STEPS_HEAD, seed=42, return_model=False):
    Xraw = df[Xcols].values.astype(float); yraw = df[ycol].values.astype(float)
    mx, sx = Xraw.mean(0), Xraw.std(0); my, sy = yraw.mean(), yraw.std()
    Xz = ((Xraw-mx)/sx).astype("float32"); yz = ((yraw-my)/sy).reshape(-1,1).astype("float32")
    tr, te = _subsample_idx(len(Xz), 42)
    ds = {"train_input":torch.tensor(Xz[tr]),"train_label":torch.tensor(yz[tr]),
          "test_input": torch.tensor(Xz[te]),"test_label": torch.tensor(yz[te])}
    m = KAN(width=width, grid=3, k=3, seed=seed, ckpt_path="./model", auto_save=False)
    # Place the spline grid on the standardized training data once, before training; with
    # KAN_DEFAULT_GRID=1 it stays at pykan's [-1,1] and the slopes are biased (Table 4).
    if os.environ.get("KAN_DEFAULT_GRID","0")!="1":
        m.update_grid_from_samples(ds["train_input"])
    t0=time.time()
    m.fit(ds, opt="Adam",  steps=ADAM_WARM, lr=0.02, update_grid=False)   # warm-up
    h=m.fit(ds, opt="LBFGS", steps=steps,    lr=0.5,  update_grid=False)   # refine
    wall=time.time()-t0
    r2_spline = r2_score(ds["test_label"], m(ds["test_input"]).detach().numpy())
    m = m.prune(); m.auto_symbolic(lib=lib, verbose=False)
    xs=[sp.Symbol(f"x_{i+1}") for i in range(len(Xcols))]
    kan_coefs, r2_kan = None, float("nan")            # network's own affine read-out
    if lib == ["x"]:                                   # affine -> OLS anchor (exact)
        A=np.column_stack([Xraw[:,i] for i in range(len(Xcols))]+[np.ones(len(df))])
        c,*_=np.linalg.lstsq(A, yraw, rcond=None)
        const=snap_half(c[-1]) if abs(c[-1])>0.05 else sp.Integer(0)
        expr=sp.simplify(sum(snap_half(c[i])*xs[i] for i in range(len(Xcols)))+const)
        pred=A@np.array([float(snap_half(c[i])) for i in range(len(Xcols))]+[float(const)])
        r2_sym=1-np.var(yraw-pred)/np.var(yraw)
        # The closed form above is linear least squares; the network's own formula is read
        # separately below, so that the KAN is judged on its own.
        try:
            es=m.symbolic_formula(var=xs)[0][0]
            kexpr=sp.expand(my+sy*es.subs({sp.Symbol(f"x_{i+1}"):(xs[i]-mx[i])/sx[i] for i in range(len(Xcols))}))
            Pk=sp.Poly(kexpr,*xs)
            kan_coefs=[float(Pk.coeff_monomial(xs[i])) for i in range(len(Xcols))]+[float(Pk.coeff_monomial(1))]
            kpred=Xraw@np.array(kan_coefs[:-1])+kan_coefs[-1]
            r2_kan=float(1-np.var(yraw-kpred)/np.var(yraw))
        except Exception as e:
            FAILURES.append((ycol+" (network read-out)", repr(e)))
            print(f"  warning: could not read the network's own formula for {ycol}: {e!r}")
    else:                                              # control / reciprocal: destandardized KAN
        try:
            es=m.symbolic_formula(var=xs)[0][0]
            expr=sp.expand(my+sy*es.subs({sp.Symbol(f"x_{i+1}"):(xs[i]-mx[i])/sx[i] for i in range(len(Xcols))}))
        except Exception as e:
            # Never substitute a placeholder: it would be reported as a recovered form.
            FAILURES.append((ycol, repr(e)))
            raise RuntimeError(
                f"symbolic export failed for {ycol}: {e!r}. This is a pykan internal issue "
                "with libraries such as ['x','1/x','x^2']; it must be reported, not hidden."
            ) from e
        # The exported symbolic expression, scored on the full data set: a different
        # quantity from the spline score.
        _f=sp.lambdify(xs, expr, "numpy")
        _pred=np.broadcast_to(_f(*[Xraw[:,i] for i in range(len(Xcols))]), yraw.shape).astype(float)
        r2_sym=float(1-np.var(yraw-_pred)/np.var(yraw))
    out=dict(expr=expr, r2_spline=float(r2_spline), r2_sym=float(r2_sym),
             kan_coefs=kan_coefs, r2_kan=r2_kan,
             hist={"train_loss":h["train_loss"],"test_loss":h["test_loss"]}, wall=float(wall), ds=ds)
    if return_model: out["model"]=m
    return out

**Cell 3.5. Engine B, logarithmic linearization and SymPy reconstruction.** `discover_log`
builds log-space features from the positive *atoms* $\{\chi,\eta,|IP{+}3EA|,|3IP{+}EA|\}$, fits
a linear KAN, snaps the edge slopes to the nearest half-integer, re-derives the prefactor as
$\exp\big(\overline{\ln y-\sum_i p_i\ln a_i}\big)$ and hands everything to SymPy. The helper
`log_ols` gives the analytic log-space OLS solution, the anchor used in 4.6.

In [ ]:
# Log-space engine. The KAN gives the representation; the exponents are anchored on the
# exact log-space least-squares solution.
chi_s, eta_s, Ap_s, Am_s = sp.symbols("chi eta A_plus A_minus", positive=True)
ATOMS = {"chi":(np.log(-mu), chi_s), "eta":(np.log(eta), eta_s),
         "Aplus":(np.log(np.abs(IP+3*EA)), Ap_s), "Aminus":(np.log(np.abs(3*IP+EA)), Am_s)}
ATOM_LIN = {"chi":-mu, "eta":eta, "Aplus":np.abs(IP+3*EA), "Aminus":np.abs(3*IP+EA)}
# (PHYS / _snap_const / _snap_half are defined in Cell 3.1)

def log_ols(feats, ycol):
    y=df[ycol].values; g=(y>1e-12)
    for f in feats: g&=np.abs(ATOM_LIN[f])>1e-9
    A=np.column_stack([np.log(np.abs(ATOM_LIN[f][g])) for f in feats]+[np.ones(g.sum())])
    c,*_=np.linalg.lstsq(A, np.log(y[g]), rcond=None); return c[:-1], c[-1]

def discover_log(name, feats, ycol, ref, steps=STEPS_HEAD, seed=42, grid=3, k=3, verbose=True,
                 return_model=False, opt=None, lr=None, noise_sigma=0.0):
    y=df[ycol].values; g=(y>1e-12)
    for f in feats: g&=np.abs(ATOM_LIN[f])>1e-9
    # exact OLS anchor: the reported exponents and closed form
    exps_ols, b = log_ols(feats, ycol)
    slopes=[_snap_half(e) for e in exps_ols]
    resid=np.log(y[g]).copy()
    for s,f in zip(slopes,feats): resid-=float(s)*np.log(np.abs(ATOM_LIN[f][g]))
    prefactor=_snap_const(float(np.exp(resid.mean())))
    closed=prefactor
    for s,f in zip(slopes,feats): closed*=ATOMS[f][1]**s
    closed=sp.simplify(closed)
    Xln=np.column_stack([np.log(np.abs(ATOM_LIN[f][g])) for f in feats]).astype(float)
    yln=np.log(y[g]).astype(float)
    if noise_sigma>0:   # multiplicative noise in the original space == additive in log-space
        yln=yln+np.random.default_rng(seed).normal(0.0,noise_sigma,yln.size)
    if len(feats) >= 2:                          # standardize only multivariate log fits
        mx,sx=Xln.mean(0),Xln.std(0); myy,syy=yln.mean(),yln.std()
    else:                                        # univariate (S): no standardization (avoids underfit)
        mx,sx=np.zeros(Xln.shape[1]),np.ones(Xln.shape[1]); myy,syy=0.0,1.0
    Xz=((Xln-mx)/sx).astype("float32"); yz=((yln-myy)/syy).reshape(-1,1).astype("float32")
    tr,te=_subsample_idx(len(Xz), seed)   # thread the seed: vary the SPLIT, not just the init
    ds={"train_input":torch.tensor(Xz[tr]),"train_label":torch.tensor(yz[tr]),
        "test_input": torch.tensor(Xz[te]),"test_label": torch.tensor(yz[te])}
    width=[len(feats),1] if len(feats)==1 else [len(feats),1,1]
    m=KAN(width=width, grid=grid, k=k, seed=seed, ckpt_path="./model", auto_save=False)
    # Place the spline grid on the standardized training data once, before training; with
    # KAN_DEFAULT_GRID=1 it stays at pykan's [-1,1] and the slopes are biased (Table 4).
    if os.environ.get("KAN_DEFAULT_GRID","0")!="1":
        m.update_grid_from_samples(ds["train_input"])
    t0=time.time()
    if opt is None:                                   # headline: Adam warm-up + L-BFGS refine
        m.fit(ds,opt="Adam",steps=ADAM_WARM,lr=0.02,update_grid=False)
        h=m.fit(ds,opt="LBFGS",steps=steps,lr=0.5,update_grid=False)
    else:                                             # optimizer study: single optimizer
        lr_ = lr if lr is not None else (0.5 if opt=="LBFGS" else 0.02)
        h=m.fit(ds,opt=opt,steps=steps,lr=lr_,update_grid=False)
    wall=time.time()-t0
    r2_log=r2_score(ds["test_label"], m(ds["test_input"]).detach().numpy())
    mp=m.prune(); mp.auto_symbolic(lib=["x"], verbose=False)
    xs=[sp.Symbol(f"x_{i+1}") for i in range(len(feats))]

    raw_expr_from_kan = None
    kan_raw = [np.nan] * len(feats)
    r2_readout = np.nan
    kan_const = np.nan

    try:
        raw_expr_from_kan = mp.symbolic_formula(var=xs)[0][0]
        raw = sp.expand(myy+syy*raw_expr_from_kan.subs({xs[i]:(sp.Symbol(f"X_{i+1}")-mx[i])/sx[i] for i in range(len(feats))}))
        P=sp.Poly(raw, *[sp.Symbol(f"X_{i+1}") for i in range(len(feats))])
        kan_raw=[float(P.coeff_monomial(sp.Symbol(f"X_{i+1}"))) for i in range(len(feats))]
        kan_const=float(P.coeff_monomial(1))          # network's own ln(prefactor)
        # Scores the symbolic formula, not the spline: separates a failed fit from a
        # failed read-out.
        _fz=sp.lambdify(xs, raw_expr_from_kan, "numpy")
        _pz=np.broadcast_to(_fz(*[Xz[te][:,i] for i in range(len(feats))]), yz[te].ravel().shape)
        r2_readout=float(1-np.var(yz[te].ravel()-_pz)/np.var(yz[te].ravel()))
    except Exception as e:
        # Keep the NaN defaults, but never swallow the failure silently.
        FAILURES.append((name+" (network read-out)", repr(e)))
        print(f"  warning: could not read the network's own formula for {name}: {e!r}")
        raw = None

    # KAN-only closed form, read from the network itself: `closed` above never touches the
    # network and this one never touches OLS.
    kan_pref, kan_closed, r2_kan_closed, r2_kan_unsnapped = np.nan, None, np.nan, np.nan
    if np.all(np.isfinite(kan_raw)) and np.isfinite(kan_const):
        kan_pref=float(np.exp(kan_const))
        _kc=_snap_const(kan_pref)
        for _p,f in zip([_snap_half(v) for v in kan_raw],feats): _kc*=ATOMS[f][1]**_p
        kan_closed=sp.simplify(_kc)
        _fk=sp.lambdify([ATOMS[f][1] for f in feats], kan_closed, "numpy")
        _pk=np.broadcast_to(_fk(*[ATOM_LIN[f] for f in feats]), y.shape).astype(float)
        r2_kan_closed=float(1-np.var(y-_pk)/np.var(y))
        _pu=kan_pref*np.prod([np.abs(ATOM_LIN[f])**p for f,p in zip(feats,kan_raw)],axis=0)
        r2_kan_unsnapped=float(1-np.var(y-_pu)/np.var(y))

    # (1) analytic log-OLS R2: is the true relationship log-linear?
    A_=np.column_stack([np.log(np.abs(ATOM_LIN[f][g])) for f in feats]+[np.ones(int(g.sum()))])
    c_,*_=np.linalg.lstsq(A_,np.log(y[g]),rcond=None); pr_=A_@c_
    r2_ols=float(1-np.var(np.log(y[g])-pr_)/np.var(np.log(y[g])))
    # (2) closed-form recovery R2 on the FULL data (this certifies recovery)
    fcl=sp.lambdify([ATOMS[f][1] for f in feats], closed, "numpy")
    pred_=np.broadcast_to(fcl(*[ATOM_LIN[f] for f in feats]), y.shape).astype(float)
    r2_closed=float(1-np.var(y-pred_)/np.var(y))
    # (3) r2_log below = internal KAN spline fit (standardized, subsample)
    out=dict(name=name, feats=feats, slopes=slopes, raw_slopes=kan_raw, ols_slopes=[float(e) for e in exps_ols],
             prefactor=prefactor, closed=closed, r2_log=float(r2_log), r2_readout=r2_readout, r2_ols=r2_ols,
             kan_const=kan_const, kan_pref=kan_pref, kan_closed=kan_closed,
             r2_kan_closed=r2_kan_closed, r2_kan_unsnapped=r2_kan_unsnapped, r2_closed=r2_closed,
             hist={"train_loss":h["train_loss"],"test_loss":h["test_loss"]}, wall=float(wall))
    if return_model: out["model"]=mp
    if verbose:
        print(f"### {name}  (ref {ref})   y = {closed}   [wall={wall:.1f}s]")
        print(f"    exponents (OLS-anchored) = {dict(zip(feats,[str(s) for s in slopes]))}   prefactor = {prefactor}")
        print(f"    R2:  analytic log-OLS = {r2_ols:.6f}  |  closed-form (all N) = {r2_closed:.6f}  |  KAN spline (std., subsample) = {r2_log:.4f}\n")
    return out

## 4. Results

### 4.1 Affine descriptors, direct recovery ($\mu,\chi,\eta$)

**Cell 4.1.** A KAN is fitted from $(IP,EA)$ with a linear library for each affine descriptor,
and the recovered expression is read back.

In [ ]:
affine_ref = {"mu":"-(IP+EA)/2","chi":"(IP+EA)/2","eta":"IP-EA"}
affine_out = {}
IP_s, EA_s = sp.symbols("IP EA")
for name,ref in affine_ref.items():
    r = kan_symbolic(["IP","EA"], name, width=[2,1,1], lib=["x"], steps=STEPS_HEAD, return_model=True)
    e = sp.nsimplify(r["expr"].subs({sp.Symbol("x_1"):IP_s, sp.Symbol("x_2"):EA_s}), rational=True, tolerance=0.02)
    r["closed"]=e; affine_out[name]=r
    print(f"{name:4s} ref={ref:12s} anchor (linear OLS) -> {e}   R2={r['r2_sym']:.6f} | "
          f"KAN spline R2={r['r2_spline']:.4f} | wall={r['wall']:.1f}s")

### 4.2 Softness: reciprocal against logarithmic recovery

The direct route is given its best chance on the simplest non-affine target, the univariate
reciprocal $S=1/\eta$, with `1/x` explicitly in the library. Read the two outputs against each
other: the direct fit is numerically excellent, yet exports a Mobius transformation in the
standardized variable, while the logarithmic route returns $1/\eta$ exactly.

In [ ]:
r_dir = kan_symbolic(["eta"],"S",width=[1,1],lib=["x","1/x"],steps=STEPS_HEAD)
print("Direct 1/x :  R2 =",round(r_dir["r2_sym"],6)," form =",sp.N(r_dir["expr"],4))
print("\nLogarithmic route:")
log_out={}
log_out["S"]=discover_log("S",["eta"],"S","1/eta",return_model=True)

### 4.3 The multiplicative problem (control experiment)

The quotient and quadratic descriptors are fitted *directly* from their natural variables, with
every advantage given to the additive KAN: a two-input network, a library containing `x`, `1/x`
and `x^2`, and the headline budget and grid placement. The spline interpolates the targets
perfectly, while its exported rational is less accurate on the full data set.

In [ ]:
def complexity(expr):
    return len(str(sp.simplify(expr))), int(sp.count_ops(expr))
raw_ctrl = {}
for name,cols in [("Nmax",["mu","eta"]),("omega",["mu","eta"])]:
    r = kan_symbolic(cols,name,width=[2,1,1],lib=["x","1/x","x^2"],steps=STEPS_HEAD)
    L,ops = complexity(r["expr"]); raw_ctrl[name]=(r["r2_sym"],L,ops)
    print(f"{name:6s} R2(spline)={r['r2_spline']:.4f} R2(symbolic)={r['r2_sym']:.4f}"
          f"  |expr|={L} chars  ops={ops}")
print("\n=> high R2 but large nested symbolic form: NOT a recovered law.")

### 4.4 Logarithmic recovery of the multiplicative descriptors

**Cell 4.4.** The same descriptors in log-space, where SymPy exponentiates the fitted log-linear
laws back to closed forms. Models are retained for the figures below.

In [ ]:
log_out["Nmax"]   = discover_log("Nmax",       ["chi","eta"],   "Nmax",       "chi/eta",            return_model=True)
log_out["omega"]  = discover_log("omega",      ["chi","eta"],   "omega",      "chi^2/(2 eta)",      return_model=True)
log_out["wplus"]  = discover_log("omega_plus", ["Aplus","eta"], "omega_plus", "A_plus^2/(16 eta)",  return_model=True)
log_out["wminus"] = discover_log("omega_minus",["Aminus","eta"],"omega_minus","A_minus^2/(16 eta)", return_model=True)

### 4.4b Prefactor recovery without snapping

**Cell 4.4b.** The raw `exp(mean residual)` before any snapping, so the magnitude of the `PHYS`
prior is on the record. The regime where snapping does real work is the noisy one of 4.12.

In [ ]:
print("Prefactor recovery WITHOUT snapping (raw exp(mean residual)):\n")
print(f"{'descriptor':12s} {'raw':>16s} {'snapped':>10s} {'exact':>10s} {'|raw-exact|':>13s}")
_FE={"S":["eta"],"Nmax":["chi","eta"],"omega":["chi","eta"],
     "wplus":["Aplus","eta"],"wminus":["Aminus","eta"]}
_YC={"S":"S","Nmax":"Nmax","omega":"omega","wplus":"omega_plus","wminus":"omega_minus"}
_exact={"S":1.0,"Nmax":1.0,"omega":0.5,"wplus":1/16,"wminus":1/16}
for key in ["S","Nmax","omega","wplus","wminus"]:
    feats=_FE[key]; ycol=_YC[key]
    y=df[ycol].values; g=(y>1e-12)
    for f in feats: g&=np.abs(ATOM_LIN[f])>1e-9
    exps,_=log_ols(feats,ycol)                       # un-snapped exponents
    resid=np.log(y[g]).copy()
    for e_,f in zip(exps,feats): resid-=float(e_)*np.log(np.abs(ATOM_LIN[f][g]))
    raw=float(np.exp(resid.mean()))
    print(f"{key:12s} {raw:16.12f} {str(log_out[key]['prefactor']):>10s} "
          f"{_exact[key]:10.6f} {abs(raw-_exact[key]):13.2e}")
print("\nThe raw prefactors already match the exact values to ~1e-12, so on this noiseless")
print("test bed the PHYS snapping library has nothing to correct. It earns its place only")
print("in the noisy regime of section 4.12.")

### 4.4c Closed forms from the network alone

**The decisive test.** Everything above this point reports the *anchored* exponents, which come
from exact least squares and would be identical had no network been trained. This cell reads the
trained network's own edge slopes and intercept, rounds them, and compares the result
symbolically with the reference. The second table is a weaker variant: the structure comes from
the network and the single constant is re-estimated.

In [ ]:
REF = {"mu": -(IP_s+EA_s)/2, "chi": (IP_s+EA_s)/2, "eta": IP_s-EA_s,
       "S": 1/eta_s, "Nmax": chi_s/eta_s, "omega": chi_s**2/(2*eta_s),
       "wplus": Ap_s**2/(16*eta_s), "wminus": Am_s**2/(16*eta_s)}
kan_only = {}
print(f"{'desc':7s} {'network (unrounded)':40s} {'rounded -> closed form':24s} {'match':5s} "
      f"{'R2 unrounded':>12s} {'R2 rounded':>10s}")
for k in ["mu","chi","eta"]:
    r=affine_out[k]; c=r["kan_coefs"]
    if c is None or not np.all(np.isfinite(c)):
        kan_only[k]=False; print(f"{k:7s} (network read-out failed)"); continue
    const=_snap_half(c[2]) if abs(c[2])>0.05 else sp.Integer(0)
    f=sp.simplify(_snap_half(c[0])*IP_s+_snap_half(c[1])*EA_s+const)
    pr=IP*float(_snap_half(c[0]))+EA*float(_snap_half(c[1]))+float(const)
    r2r=float(1-np.var(df[k].values-pr)/np.var(df[k].values))
    ok=sp.simplify(f-REF[k])==0; kan_only[k]=ok
    raw=f"IP:{c[0]:+.4f} EA:{c[1]:+.4f} c:{c[2]:+.4f}"
    print(f"{k:7s} {raw:40s} {str(f):24s} {'yes' if ok else 'NO':5s} {r['r2_kan']:12.6f} {r2r:10.6f}")
for k in ["S","Nmax","omega","wplus","wminus"]:
    r=log_out[k]
    if r["kan_closed"] is None:
        kan_only[k]=False; print(f"{k:7s} (network read-out failed)"); continue
    ok=sp.simplify(r["kan_closed"]-REF[k])==0; kan_only[k]=ok
    raw="p:"+",".join(f"{v:+.4f}" for v in r["raw_slopes"])+f" c:{r['kan_pref']:.5f}"
    print(f"{k:7s} {raw:40s} {str(r['kan_closed']):24s} {'yes' if ok else 'NO':5s} "
          f"{r['r2_kan_unsnapped']:12.6f} {r['r2_kan_closed']:10.6f}")
n_ok=sum(kan_only.values())
print(f"\nThe network alone, after rounding, recovers {n_ok}/8 CRDFT closed forms.")
if n_ok<8: print("Not recovered by the network alone: "+", ".join(k for k,v in kan_only.items() if not v))
if FAILURES: print("Read-out failures recorded:", FAILURES)

# ---- Structure from the network, constant re-estimated -----------------------------
# Not the anchor: the exponents come from the network and only the constant is refitted.
print("\nStructure from the network, constant re-estimated with the structure fixed:")
print(f"{'desc':7s} {'network exponents (rounded)':28s} {'refit constant':>15s} {'closed form':24s} match")
kan_struct = {k: kan_only[k] for k in ["mu","chi","eta"]}
_FE={"S":["eta"],"Nmax":["chi","eta"],"omega":["chi","eta"],"wplus":["Aplus","eta"],"wminus":["Aminus","eta"]}
_YC={"S":"S","Nmax":"Nmax","omega":"omega","wplus":"omega_plus","wminus":"omega_minus"}
for k in ["S","Nmax","omega","wplus","wminus"]:
    r=log_out[k]
    if not np.all(np.isfinite(r["raw_slopes"])):
        kan_struct[k]=False; print(f"{k:7s} (network read-out failed)"); continue
    ph=[_snap_half(v) for v in r["raw_slopes"]]
    y=df[_YC[k]].values; g=(y>1e-12)
    for f in _FE[k]: g&=np.abs(ATOM_LIN[f])>1e-9
    res=np.log(y[g])-sum(float(p)*np.log(np.abs(ATOM_LIN[f][g])) for p,f in zip(ph,_FE[k]))
    c_fit=float(np.exp(res.mean())); cs=_snap_const(c_fit)
    form=cs
    for p,f in zip(ph,_FE[k]): form*=ATOMS[f][1]**p
    form=sp.simplify(form); ok=sp.simplify(form-REF[k])==0; kan_struct[k]=ok
    print(f"{k:7s} {str([str(p) for p in ph]):28s} {c_fit:15.6f} {str(form):24s} {'yes' if ok else 'NO'}")
n_st=sum(kan_struct.values())
print(f"\nWith the structure from the network and the constant re-estimated: {n_st}/8 recovered.")

### 4.5 Per-descriptor KAN diagrams

**Cell 4.5 (Figure 1).** The *pruned, symbolic* KAN for every descriptor: nodes are variables,
edges carry the learned univariate activation. In panels (g)-(h), $A^{+}=|IP+3EA|$ and
$A^{-}=|3IP+EA|$.

In [ ]:
# Figure 1: all eight pruned KANs as panels (a)-(h) of ONE figure, at RSC column width.
import rsc_style as rsc, figures_rsc as F
rsc.use()
F.fig01_kan_diagrams(affine_out, log_out);

### 4.6 Exponent recovery: the network against the analytic anchor

Deviations of both routes from the exact exponents. The anchor solves the linear problem
exactly, so its column is a reference rather than a result. The figure plots absolute deviations
logarithmically; pass `scale="linear"` for the signed version.

In [ ]:
ref_exp = {"S":(None,-1),"Nmax":(1,-1),"omega":(2,-1),"wplus":(2,-1),"wminus":(2,-1)}
feat_of = {"S":["eta"],"Nmax":["chi","eta"],"omega":["chi","eta"],
           "wplus":["Aplus","eta"],"wminus":["Aminus","eta"]}
ycol_of = {"S":"S","Nmax":"Nmax","omega":"omega","wplus":"omega_plus","wminus":"omega_minus"}

# The KAN column is raw_slopes (the network's estimate), the OLS column the analytic anchor.
print(f"{'desc':7s} {'atom':7s} {'ref':>5s} {'KAN(raw)':>9s} {'dev':>8s} {'OLS':>9s} {'dev':>10s}")
for key in ["S","Nmax","omega","wplus","wminus"]:
    feats=feat_of[key]; ols_s,_=log_ols(feats,ycol_of[key]); res=log_out[key]
    for f,rs,os_ in zip(feats,res["raw_slopes"],ols_s):
        rf = ref_exp[key][0] if f!="eta" else ref_exp[key][1]
        print(f"{key:7s} {f:7s} {str(rf):>5s} {rs:9.4f} {rs-rf:+8.4f} {os_:9.4f} {os_-rf:+10.2e}")

_dev={}
for key in ["S","Nmax","omega","wplus","wminus"]:
    for f,rs in zip(feat_of[key],log_out[key]["raw_slopes"]):
        rf = ref_exp[key][0] if f!="eta" else ref_exp[key][1]
        _dev[(key,f)] = abs(rs-rf)/abs(rf)
_w=max(_dev,key=_dev.get)
print("\nThe anchor reproduces the reference to ~1e-14 throughout, as it must on a noiseless")
print(f"monomial target. The network's own slopes deviate by at most "
      f"{100*_dev[_w]:.2f}% ({_w[0]}, exponent of {_w[1]}).")

F.fig02_exponents(log_out, log_ols, feat_of, ycol_of, ref_exp);

### 4.7 Numerical validation of the recovered closed forms

**Cell 4.7 (Figure 3).** The *symbolic* closed forms are substituted back into the full dataset
and scored against the true descriptor columns. Points on the diagonal indicate exact recovery.

In [ ]:
val = {
 "mu":   (affine_out["mu"]["closed"],   {IP_s:IP,EA_s:EA},                 "mu"),
 "chi":  (affine_out["chi"]["closed"],  {IP_s:IP,EA_s:EA},                 "chi"),
 "eta":  (affine_out["eta"]["closed"],  {IP_s:IP,EA_s:EA},                 "eta"),
 "S":    (log_out["S"]["closed"],       {eta_s:eta},                       "S"),
 "Nmax": (log_out["Nmax"]["closed"],    {chi_s:-mu,eta_s:eta},             "Nmax"),
 "omega":(log_out["omega"]["closed"],   {chi_s:-mu,eta_s:eta},             "omega"),
 "wplus":(log_out["wplus"]["closed"],   {Ap_s:np.abs(IP+3*EA),eta_s:eta},  "omega_plus"),
 "wminus":(log_out["wminus"]["closed"], {Am_s:np.abs(3*IP+EA),eta_s:eta},  "omega_minus"),
}
final_r2={}
for k,(expr,sub,col) in val.items():
    f=sp.lambdify(list(sub.keys()),expr,"numpy")
    pred=np.broadcast_to(f(*sub.values()),df[col].values.shape).astype(float)
    final_r2[k]=r2_score(df[col].values,pred)
print("Closed-form R2 on all N (this certifies the ANCHOR, not the network):")
print({k:round(v,6) for k,v in final_r2.items()})

# Figure 3's scatter layer is subsampled for display; the R2 above uses the full data set.
F.fig03_parity(df, val);

### 4.8 Learning curves (all descriptors)

**Cell 4.8 (Figure 4).** Train/test RMSE against optimizer step for every headline fit.
Overlapping train and test traces indicate no overfitting.

In [ ]:
F.fig04_learning_curves(affine_out, log_out);

### 4.9 Optimizer study: L-BFGS against Adam

**L-BFGS** is compared against **Adam** on a bivariate ($\omega$) and a univariate ($S$)
descriptor, at the headline budget and with a doubled Adam budget. Results are averaged over the
five seeds and given as mean plus or minus standard deviation, because a single-seed comparison
of these two optimizers is not stable. Where the spread is comparable to the mean the comparison
is not resolved at that budget, and the ratios, not the absolute timings, carry over.

In [ ]:
# Averaged over SEEDS: a single-seed comparison of the two optimizers is not stable.
opt_desc = {"omega":(["chi","eta"],"omega"), "S":(["eta"],"S")}
opt_rows=[]; opt_hist={}
for key,(feats,ycol) in opt_desc.items():
    for _opt, _budgets in [("LBFGS",[LBFGS_OPT_STEPS]), ("Adam", ADAM_BUDGETS)]:
        for b in _budgets:
            _rmse, _wall = [], []
            for _sd in SEEDS:
                _r = discover_log(key,feats,ycol,"-",steps=b,opt=_opt,seed=_sd,verbose=False)
                _rmse.append(_r["hist"]["test_loss"][-1]); _wall.append(_r["wall"])
                if _sd == SEEDS[0]:
                    opt_hist[(key,_opt,b)] = _r["hist"]     # curves: first seed
            _rmse = np.asarray(_rmse, dtype=float)
            # 5-tuple kept for figures_rsc.fig05_optimizer; the spreads follow it.
            opt_rows.append((key,_opt,b,float(_rmse.mean()),float(np.mean(_wall)),
                             float(_rmse.std()), len(SEEDS)))

print(f"{'desc':6s} {'opt':6s} {'steps':>5s} {'test_RMSE (mean+/-sd)':>26s} "
      f"{'wall(s)':>8s} {'seeds':>6s}")
for r in opt_rows:
    print(f"{r[0]:6s} {r[1]:6s} {r[2]:5d} {r[3]:13.3e} +/- {r[5]:8.1e} {r[4]:8.1f} {r[6]:6d}")

print()
for key in opt_desc:
    _L = [r for r in opt_rows if r[0]==key and r[1]=="LBFGS"][0]
    for _A in [r for r in opt_rows if r[0]==key and r[1]=="Adam"]:
        print(f"{key:6s} Adam {_A[2]:4d} steps is {_A[3]/_L[3]:5.1f}x less accurate than "
              f"L-BFGS {_L[2]} steps, at {_A[4]/_L[4]:4.2f}x the wall-clock")

F.fig05_optimizer(opt_hist, opt_rows);

### 4.10 Spline-grid and order sensitivity

The quantity swept is the network's own raw slopes, since the anchored exponents are identical
in every row by construction. Where training reaches a poor spline fit the read-off exponents
are meaningless, so a high internal $R^2$ is necessary but not sufficient for recovery.

In [ ]:
# The swept quantity is r["raw_slopes"]; r["slopes"] is the anchor, independent of (grid, k).
gk_rows=[]
for g,k in GRIDK:
    r = discover_log("omega",["chi","eta"],"omega","-",steps=STEPS_HEAD,grid=g,k=k,verbose=False)
    gk_rows.append((g, k,
                    float(r["slopes"][0]), float(r["slopes"][1]),      # OLS anchor (constant)
                    str(r["prefactor"]), r["r2_log"],
                    float(r["raw_slopes"][0]), float(r["raw_slopes"][1]),    # KAN (varies)
                    r["r2_readout"]))                                        # KAN symbolic read-out

print(f"{'grid':>4s} {'k':>2s} | {'p_chi(KAN)':>10s} {'p_eta(KAN)':>10s} | {'R2 spline':>9s} "
      f"{'R2 read-out':>11s} | {'OLS anchor':>12s}")
for g,k,pc,pe,pf,r2,rc,re_,rr in gk_rows:
    print(f"{g:4d} {k:2d} | {rc:10.4f} {re_:10.4f} | {r2:9.4f} {rr:11.4f} | "
          f"({pc:.0f}, {pe:.0f}, {pf})")

_ok  = [r for r in gk_rows if abs(r[6]-2)<0.25 and abs(r[7]+1)<0.25]
_bad = [r for r in gk_rows if r not in _ok]
print(f"\nExponents within 0.25 of the exact values in {len(_ok)}/{len(gk_rows)} settings.")
if _ok:
    print(f"  those settings: spline R2 {min(r[5] for r in _ok):.4f}-{max(r[5] for r in _ok):.4f}")
if _bad:
    print(f"  failed settings: " + ", ".join(f"g{r[0]}k{r[1]} (spline R2 {r[5]:.4f})" for r in _bad))
    print("  The network's exponents are only as good as its fit: where training reaches a poor")
    print("  spline R2, the read-off exponents are meaningless. The anchored OLS columns are")
    print("  identical in every row by construction and are shown for reference only.")

F.fig06_spline_stability(gk_rows);

### 4.11 Seed robustness

Five seeds, varying both the network initialization and the training subsample. Only the
network's slopes carry information here; the anchored triple is seed-independent by construction
and is printed for reference.

In [ ]:
# r["slopes"] and r["prefactor"] come from least squares, so the study reports raw_slopes.
print("omega = chi^2/(2 eta): exponents recovered per seed\n")
print(f"{'seed':>5s} {'p_chi(KAN)':>11s} {'p_eta(KAN)':>11s} {'anchored':>18s}")
_rows=[]
for s in SEEDS:
    r=discover_log("omega",["chi","eta"],"omega","-",steps=STEPS_HEAD,seed=s,verbose=False)
    _rows.append((s, r["raw_slopes"][0], r["raw_slopes"][1]))
    print(f"{s:5d} {r['raw_slopes'][0]:11.4f} {r['raw_slopes'][1]:11.4f} "
          f"{'('+str(r['slopes'][0])+', '+str(r['slopes'][1])+', '+str(r['prefactor'])+')':>18s}")

_pc=np.array([q[1] for q in _rows]); _pe=np.array([q[2] for q in _rows])
print(f"\nKAN slopes over {len(SEEDS)} seeds: p_chi = {np.nanmean(_pc):.4f} +/- {np.nanstd(_pc):.4f} "
      f"(exact 2),  p_eta = {np.nanmean(_pe):.4f} +/- {np.nanstd(_pe):.4f} (exact -1)")
print("The anchored triple is seed-independent by construction; it is shown as a reference,")
print("not as evidence of robustness.")

### 4.11b Training-set size

$\omega$ and $\Delta N_{max}$ are refitted at a range of training-set sizes with everything else
fixed, reporting the network's own exponents, the accuracy of the recovered form and the cost.
The targets are exact identities, so a held-out fraction measures convergence rather than
generalization. Set `INCLUDE_80PCT = True` to add the 80% point (105,100 molecules).

In [ ]:
# N_FIT is read by _subsample_idx at call time, so it is rebound around each fit.
INCLUDE_80PCT = False          # True adds the 80% split (105,100); slow

SIZES = [500, 1000, 3000, 10000, 20000]
if INCLUDE_80PCT:
    SIZES = SIZES + [int(0.8 * N_MOL)]

_N_FIT_SAVED = N_FIT
SWEEP_DESC = {"Nmax": (["chi", "eta"], "Nmax", "chi/eta", 1, -1),
              "omega": (["chi", "eta"], "omega", "chi^2/(2 eta)", 2, -1)}
# Reference forms in the same symbols the log engine uses (Cell 3.5).
REF_SWEEP = {"Nmax": chi_s / eta_s, "omega": chi_s**2 / (2 * eta_s)}

size_rows = []
try:
    for _name, (_feats, _ycol, _ref, _nref, _eref) in SWEEP_DESC.items():
        for _n in SIZES:
            if _n + 2000 > N_MOL:
                print(f"  skipping {_n}: no room for a disjoint 2000-molecule test set")
                continue
            N_FIT = _n                      # rebind the global read by _subsample_idx
            _r = discover_log(_name, _feats, _ycol, _ref, steps=STEPS_HEAD,
                              seed=42, verbose=False)
            _pn, _pe = _r["raw_slopes"][0], _r["raw_slopes"][1]
            size_rows.append(dict(name=_name, n_fit=_n, p_num=_pn, p_eta=_pe,
                                  num_ref=_nref, eta_ref=_eref, wall=_r["wall"],
                                  r2_spline=_r["r2_log"],
                                  r2_closed=_r["r2_kan_closed"],
                                  match=bool(sp.simplify(
                                      _r["kan_closed"] - REF_SWEEP[_name]) == 0)))
finally:
    N_FIT = _N_FIT_SAVED                    # always restore, even on an exception

print(f"{'descriptor':11s} {'N_fit':>7s} {'p_num':>9s} {'p_eta':>9s} "
      f"{'|dev|max':>9s} {'R2 spline':>10s} {'R2 closed':>10s} {'wall/s':>8s} {'match':>6s}")
for _r in size_rows:
    _dev = max(abs(_r["p_num"] - _r["num_ref"]), abs(_r["p_eta"] - _r["eta_ref"]))
    print(f"{_r['name']:11s} {_r['n_fit']:7d} {_r['p_num']:9.4f} {_r['p_eta']:9.4f} "
          f"{_dev:9.1e} {_r['r2_spline']:10.4f} {_r['r2_closed']:10.6f} {_r['wall']:8.1f} {str(_r['match']):>6s}")

# The first fit of the sweep carries one-off warm-up cost, so it is excluded.
print()
for _nm in SWEEP_DESC:
    _rs = sorted((r for r in size_rows if r["name"] == _nm), key=lambda r: r["n_fit"])[1:]
    if len(_rs) >= 2:
        _x = np.log([r["n_fit"] for r in _rs]); _y = np.log([r["wall"] for r in _rs])
        _slope = float(np.polyfit(_x, _y, 1)[0])
        print(f"{_nm:11s} cost scales as size^{_slope:.2f} "
              f"({_rs[0]['wall']:.1f}s at {_rs[0]['n_fit']} -> "
              f"{_rs[-1]['wall']:.1f}s at {_rs[-1]['n_fit']}); "
              "1.00 would be exactly linear")
print("Every recovered closed form R2 above is evaluated on all "
      f"{N_MOL} molecules, not on a held-out fraction.")

F.fig_trainsize(size_rows);

### 4.12 Noise robustness

Log-normal multiplicative noise on the target, applied as additive Gaussian noise on $\ln y$,
with five seeds per level and the headline training protocol. The same noise realizations are
reused across levels (common random numbers). The **unrounded** slopes are reported, because
snapping would map a degraded exponent back onto the exact value.

In [ ]:
# Raw, un-snapped exponents: snapping maps anything within 0.25 of the truth onto it.
NOISE_DESCS = [("Nmax", ["chi","eta"], 1, -1), ("omega", ["chi","eta"], 2, -1)]

noise_summary = {}
for name, feats, num_ref, eta_ref in NOISE_DESCS:
    nm, ns, em, es = [], [], [], []
    for sg in SIGMAS:
        nums, etas = [], []
        for sd in NOISE_SEEDS:
            r = discover_log(name, feats, name, "-", steps=STEPS_HEAD, seed=sd,
                             verbose=False, noise_sigma=sg)
            nums.append(r["raw_slopes"][0]); etas.append(r["raw_slopes"][1])
        nums, etas = np.asarray(nums), np.asarray(etas)
        nm.append(np.nanmean(nums)); ns.append(np.nanstd(nums))
        em.append(np.nanmean(etas)); es.append(np.nanstd(etas))
    noise_summary[name] = dict(sigmas=list(SIGMAS), num_mean=nm, num_std=ns,
                               eta_mean=em, eta_std=es, num_ref=num_ref, eta_ref=eta_ref)
    print(f"{name}  (exact: p_num={num_ref}, p_eta={eta_ref})")
    print(f"{'sigma':>7s} {'p_num':>18s} {'p_eta':>18s}")
    for s, a, sa, b, sb in zip(SIGMAS, nm, ns, em, es):
        print(f"{s:7.2f} {a:+9.4f} +/-{sa:6.4f} {b:+9.4f} +/-{sb:6.4f}")
    print()

import figures_rsc as F   # journal-format figure; see rsc_style.py
F.fig07_noise(noise_summary);

### 4.13 Summary table

**Cell 4.13.** Reference form, recovered closed form and validation $R^2$ for all eight
descriptors. The forms listed are the anchor's; the network alone reaches the same eight (4.4c).

In [ ]:
summary=[
 ("mu","-(IP+EA)/2",affine_out["mu"]["closed"],final_r2["mu"]),
 ("chi","(IP+EA)/2",affine_out["chi"]["closed"],final_r2["chi"]),
 ("eta","IP-EA",affine_out["eta"]["closed"],final_r2["eta"]),
 ("S","1/eta",log_out["S"]["closed"],final_r2["S"]),
 ("Nmax","chi/eta",log_out["Nmax"]["closed"],final_r2["Nmax"]),
 ("omega","chi^2/(2 eta)",log_out["omega"]["closed"],final_r2["omega"]),
 ("omega_plus","|IP+3EA|^2/(16 eta)",log_out["wplus"]["closed"],final_r2["wplus"]),
 ("omega_minus","|3IP+EA|^2/(16 eta)",log_out["wminus"]["closed"],final_r2["wminus"]),
]
print(f"{'descriptor':12s} {'reference':22s} {'recovered':22s} {'R2':>9s}")
for n,ref,rec,r2 in summary:
    print(f"{n:12s} {ref:22s} {str(rec):22s} {r2:9.6f}")

### 4.14 Persist everything (so a lost session never costs another run)

This cell writes `results/kan_paper_numbers.json` (every quantity the paper cites) and
`results/figure_inputs.pkl` (the objects the plotting functions consume, with live models and
training tensors stripped). Keep the whole `figures/` folder, including `figures/panels/`:
`regenerate_figures.py` then rebuilds every figure without re-fitting.

In [ ]:
import json, pickle, os
os.makedirs("results", exist_ok=True)

def _clean(d):
    """Drop what cannot outlive the session: live models and training tensors."""
    return {k: {kk: vv for kk, vv in v.items() if kk not in ("model", "ds")}
            for k, v in d.items()}

# ---- 1. every number the manuscript cites, machine-readable ------------------
_paper = {
    "n_molecules": int(N_MOL),
    "csv_sha256": DATA_SHA256,
    "used_synthetic": bool(USED_SYNTHETIC),
    "split": {"n_fit": int(N_FIT), "n_test": 2000,
              "n_unused": int(N_MOL - N_FIT - 2000)},
    "env": {"python": platform.python_version(),
            "torch": _ver("torch"), "pykan": _ver("pykan"),
            "sympy": _ver("sympy"), "numpy": _ver("numpy")},
    "affine": {k: {"closed_form": str(v["expr"]), "r2_closed": v["r2_sym"],
                   "r2_spline": v["r2_spline"], "kan_coefs": v.get("kan_coefs"),
                   "r2_kan": v.get("r2_kan"), "wall_s": v["wall"]}
               for k, v in affine_out.items()},
    "log": {k: {"closed_form": str(v["closed"]),
                "kan_closed_form": str(v.get("kan_closed")),
                "anchored_exponents": [str(x) for x in v["slopes"]],
                "network_exponents": v["raw_slopes"],
                "prefactor": str(v["prefactor"]), "kan_prefactor": v.get("kan_pref"),
                "r2_ols": v["r2_ols"], "r2_closed": v["r2_closed"],
                "r2_spline": v["r2_log"], "r2_kan_closed": v.get("r2_kan_closed"),
                "wall_s": v["wall"]}
            for k, v in log_out.items()},
    "optimizer_rows": [list(r) for r in opt_rows],
    "spline_rows": [list(r) if isinstance(r, (list, tuple)) else r for r in gk_rows],
    "seeds": SEEDS,
}
for _name, _obj in [("noise_summary", "noise_summary"), ("size_rows", "size_rows")]:
    if _obj in globals():
        _paper[_name] = globals()[_obj]

with open("results/kan_paper_numbers.json", "w") as _f:
    json.dump(_paper, _f, indent=2, default=str)

# ---- 2. the figure inputs, models stripped ----------------------------------
_figs = {"affine_out": _clean(affine_out), "log_out": _clean(log_out),
         "opt_hist": {str(k): v for k, v in opt_hist.items()},
         "opt_rows": [list(r) for r in opt_rows], "gk_rows": gk_rows,
         "ref_exp": ref_exp, "feat_of": feat_of, "ycol_of": ycol_of}
for _o in ("noise_summary", "size_rows"):
    if _o in globals():
        _figs[_o] = globals()[_o]
with open("results/figure_inputs.pkl", "wb") as _f:
    pickle.dump(_figs, _f)

print("wrote results/kan_paper_numbers.json  "
      f"({os.path.getsize('results/kan_paper_numbers.json')/1024:.1f} kB)")
print("wrote results/figure_inputs.pkl       "
      f"({os.path.getsize('results/figure_inputs.pkl')/1024:.1f} kB)")
print("\nDownload results/ AND the whole figures/ folder (including figures/panels/).")
print("regenerate_figures.py then rebuilds every figure without re-fitting.")